In [1]:
import re
import json
import faiss    # pip install sentence-transformers faiss-cpu
import numpy as np
from sentence_transformers import SentenceTransformer


def clean_prompt(prompt):
    # Split the prompt into comma-separated fields
    parts = prompt.split(',')

    cleaned_parts = []

    for part in parts:
        # Remove the entire field if it contains "lora"
        if 'lora' in part.lower():
            continue

        cleaned_parts.append(part.strip())

    # Rebuild the prompt
    prompt = ', '.join(cleaned_parts)

    # Normalize whitespace
    prompt = re.sub(r'\s+', ' ', prompt)

    # Remove trailing commas/spaces
    prompt = prompt.rstrip(' ,')

    return prompt

def search(query, k=5):
    """
    Retrieve the k most semantically similar prompts from the FAISS index.

    Args:
        query (str):
            Natural language query describing the desired image.

        k (int, optional):
            Number of results to retrieve.
            Defaults to 5.

    Returns:
        list[dict]:
            A list of dictionaries containing:

            - id (int): Original prompt identifier.
            - score (float): Cosine similarity score.
            - prompt (str): Retrieved prompt text.
    """

    # Convert the query into an embedding vector.
    # normalize_embeddings=True ensures that the resulting vector has unit length, allowing cosine similarity to be computed
    # using an inner product.
    query_embedding = model.encode([query], normalize_embeddings=True)

    # FAISS expects float32 vectors.
    query_embedding = np.array(query_embedding, dtype="float32")

    # Perform a nearest-neighbor search.
    #
    # scores:
    #     Similarity values for each retrieved prompt.
    #
    # indices:
    #     Positions of the matching prompts inside the 'items' list.
    scores, indices = index.search(query_embedding, k)

    results = []

    # Reconstruct the original prompt information from the returned indices.
    for score, idx in zip(scores[0], indices[0]):

        item = items[idx]

        results.append({"id": item["id"], "score": float(score), "prompt": item["prompt"]})

    return results

def build_rag_examples(results):
    """
    Convert retrieved search results into a text block that can be inserted into a prompt template.

    Args:
        results (list[dict]):
            Output produced by the search() function.

    Returns:
        str:
            Formatted examples suitable for inclusion in an LLM prompt.
    """

    examples = []

    for i, result in enumerate(results, start=1):

        examples.append(f"Example {i}:\n{result['prompt']}")

    return "\n\n".join(examples)

In [2]:
# Load a pre-trained embedding model. This model converts text into dense vectors that capture semantic meaning.
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# Load all prompts from a JSON file. Expected format: 
# [{"id": <id1>, "prompt": <prompt1>}, {"id": <id2>, "prompt": <prompt2>}, ...]
#
# The 'id' field is not used by FAISS itself, but it is useful for identifying prompts later when search results are returned.
with open("prompts.json", "r", encoding="utf-8") as f:
    items = json.load(f)

# Clean prompts.
items = [{"id": prompt_id, "prompt": clean_prompt(items[prompt_id])} for prompt_id in items]

In [4]:
# Extract the text that will be embedded. Each prompt becomes a single document in the vector database. The order is important.
#
# items[0] <-> embeddings[0]
# items[1] <-> embeddings[1]
# items[2] <-> embeddings[2]
#
# This mapping allows us to recover the original prompt after a search.
texts = []

for item in items:
    prompt = item.get("prompt", "")
    texts.append(prompt)

In [6]:
# Convert all prompts into embedding vectors. Only execute this to cretae the embedding (not for loading it).
#
# normalize_embeddings=True:
#     Normalizes each vector to length 1.
#     This makes cosine similarity equivalent to a dot product.
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

# FAISS requires float32 vectors.
embeddings = np.array(embeddings, dtype="float32")

# Save the embeddings as a file.
np.save("embeddings.npy", embeddings)

Batches:   0%|          | 0/123 [00:00<?, ?it/s]

In [7]:
# Load the embedding.
embeddings = np.load("embeddings.npy")

In [8]:
# Create a FAISS index using Inner Product (IP) similarity.
# Since all vectors were normalized above, Inner Product == Cosine Similarity.
# embeddings.shape[1] is the embedding dimension. For all-MiniLM-L6-v2 this is typically 384.
index = faiss.IndexFlatIP(embeddings.shape[1])

In [9]:
# Insert all embedding vectors into the index.
#
# After this step, FAISS can perform fast similarity searches.
#
# Vector 0 corresponds to items[0]
# Vector 1 corresponds to items[1]
# Vector 2 corresponds to items[2]
# etc.
index.add(embeddings)

print(f"Indexed {len(items)} prompts.")

Indexed 3928 prompts.


In [10]:
# Testing.
query = "Hi there! Make an image of a cyberpunk girl standing in neon rain. Thats it!"

# Retrieve the five most similar prompts.
results = search(query=query, k=5)

# Examples from RAG ready to plug in the main prompt.
examples = build_rag_examples(results)
for x in examples.split('\n\n'):
    print(x + '\n')

Example 1:
a cyberpunk street full of neon lights and colorful holograms beautiful landscape, dusk, woman, very intricate, very detailed, sharp, bright, colorful

Example 2:
a cyberpunk woman with mechanical arms, robotic legs, and a chrome neck collar wearing glowing cyan visor glasses and an iridescent silver bomber jacket, sitting relaxed on a worn pink velvet sofa, surrounded by pink flamingos and tropical plants, large retro synthwave sunset mural on the wall behind her, pink and magenta neon ambient lighting, vaporwave aesthetic, cinematic composition, hyper-detailed 3D render, 8k

Example 3:
cyberpunk world beautiful landscape, evening, woman, very intricate, very detailed, sharp, bright, colorful

Example 4:
a cyberpunk street full of futuristic technology beautiful landscape, bright luminous night, woman, very intricate, very detailed, sharp, bright, colorful, cyberpunk

Example 5:
a futuristic cyberpunk club beautiful landscape, rainy day, woman, very intricate, very detailed